# 07 — Flow and water-quality drivers

Assemble the covariates that might explain the trends: flow regime, mining-legacy water quality, and drought.

**Reads** gauge records (NWIS), WQP, trends from `06`  
**Writes** flow metric table, water-quality inventory, association results  
**Status** Phase 5 — skeleton

> Skeleton. Section headings and the config cell are in place; the analysis cells are deliberately empty for the group to fill in together.

## 0. Setup

In [ ]:
import os, sys

# PROJ/GDAL paths must be set before any geospatial import: the Jupyter kernel starts
# without `conda activate`, so PROJ cannot otherwise find its database.
def _find_share(name):
    for base in (sys.prefix, sys.base_prefix):
        p = os.path.join(base, "share", name)
        if os.path.isdir(p):
            return p
    return None

_proj, _gdal = _find_share("proj"), _find_share("gdal")
if _proj:
    os.environ["PROJ_DATA"] = os.environ["PROJ_LIB"] = _proj
if _gdal:
    os.environ.setdefault("GDAL_DATA", _gdal)

import json
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import dataretrieval.nwis as nwis

def _repo_root():
    """Walk up from the working directory to the repo root."""
    here = Path.cwd().resolve()
    for p in (here, *here.parents):
        if (p / ".git").exists() or (p / "environment.yml").is_file():
            return p
    raise RuntimeError("Could not find the repo root from " + str(here))

REPO     = _repo_root()
DATA_DIR = REPO / "data"
RUNS_DIR = REPO / "runs"          # manifests live here because data/ is gitignored
RUNS_DIR.mkdir(parents=True, exist_ok=True)

# ---- Figures ----
# figures/ is gitignored, but tracked via .gitkeep so it exists on a fresh clone —
# nothing to set up after cloning. Contents stay out of git because the repo is
# public and an executed run embeds a 60 cm gallery map (research plan §11).
FIG_DIR = REPO / "figures"
FIG_DPI = 300

def savefig(name, dpi=FIG_DPI):
    """Save the current figure to figures/<FIG_SUBDIR>/<name>.png.

    Call this BEFORE plt.show(): showing a figure can clear it, and you would
    silently save a blank page. FIG_SUBDIR is set in the config cell.
    """
    out = FIG_DIR / FIG_SUBDIR
    out.mkdir(parents=True, exist_ok=True)
    path = out / f"{name}.png"
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    print(f"figure -> {path.relative_to(REPO)}")
    return path

print("Imports OK")
print(f"  repo : {REPO}")

## 1. Configuration

Every parameter lives here. Pointing this notebook at another tile or another river is a single-cell edit.

In [ ]:
# ---- The pilot tile ----
TILE = "13TFJ"                       # holds Angostura, Buffalo Gap, Red Shirt, Scenic

# ---- Example reaches: one window per 8-digit USGS gauge inside 13TFJ ----
# Gauge-anchored so every window has a flow record to read alongside it (notebook 07).
# These are the walkthrough reaches, NOT the full corridor -- scaling is Phase 6.
EXAMPLE_WINDOWS = [
    {"site_no": "06401500", "name": "Angostura",   "lon": -103.4340, "lat": 43.3470},
    {"site_no": "06402600", "name": "Buffalo Gap", "lon": -103.2350, "lat": 43.4230},
    {"site_no": "06403700", "name": "Red Shirt",   "lon": -102.8921, "lat": 43.6724},
    {"site_no": "06408650", "name": "Scenic",      "lon": -102.5500, "lat": 43.7800},
]
WINDOW_HALF_M = 1000                 # half-width -> 2 x 2 km windows, as in notebook 03
# VERIFY: lon/lat for all but Red Shirt are approximate -- replace from the
# usgs_gauges layer of cheyenne_corridor_aoi.gpkg on first run.

# ---- Gauges ----
# Only 8-digit ids are surface-water gauges. The usgs_gauges layer holds 275 NLDI sites but
# 224 are 15-digit groundwater/misc sites with no discharge record -- filter FIRST.
GAUGES = [w["site_no"] for w in EXAMPLE_WINDOWS]

# ---- Record completeness ----
# Angostura (06401500) is seasonal, ~181 days/yr since 1978: a "complete calendar year"
# filter silently discards a continuous 1945-2024 record. Require years per day-of-year.
MIN_YEARS_PER_DOY = 10

# ---- Flow metrics ----
# Cottonwood recruitment tracks flood timing and recession rate, not annual mean flow.
FLOW_METRICS = ["annual_peak", "peak_doy", "recession_rate", "n_day_min_7", "n_day_min_30"]

# ---- Water quality ----
WQP_CHARACTERISTICS = ["Uranium", "Radium", "Arsenic", "Selenium", "Sulfate",
                       "Total dissolved solids"]

# ---- Upstream ----
CONDITION_RUN = "condition_phenology_vbet_13TFJ"

# ---- Outputs ----
RUN_NAME = "drivers_13TFJ"
FIG_SUBDIR = RUN_NAME               # figures/<run>/
OUT_DIR  = DATA_DIR / RUN_NAME

## 2. Gauge records

Pull daily discharge for the four window gauges. Median flow below Angostura is ~2 cfs against ~65 cfs at Buffalo Gap just downstream — the dam effect is the loudest signal in this corridor and the windows are placed to straddle it.

## 3. Flow metrics

Per-year metrics from `FLOW_METRICS`. Apply the completeness rule, not an outlier filter — Eagle Butte's 2008 spike is a genuine 65,200 cfs flood in a two-year partial record.

## 4. Water-quality inventory

What WQP actually holds near the corridor, by station and analyte. **This is an inventory first:** §17 flags that coverage may only support a contextual account, not a formal analysis. Decide which after looking, and say which one you did.

## 5. Drought covariates

PDSI / SPEI for the corridor climate division.

## 6. Association

Relate patch trends from `06` to flow and drought. **Inference honesty (§8.3):** these are observational associations on a single river with no control reach. Do not write them up as causal attribution.

## 7. Save and record the run

Every output gets a manifest in `runs/` — small, text, always committed, even when the raster it describes is not.

In [ ]:
manifest = {
    "run_name":    RUN_NAME,
    "notebook":    "07_Flow_and_WaterQuality_Drivers.ipynb",
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "inputs":      {},          # STAC item IDs, upstream run names, source manifests
    "parameters":  {},          # everything from the config cell
    "environment": {"python": sys.version.split()[0]},
    "results":     {},
    "outputs":     [],
}

# manifest_path = RUNS_DIR / f"{RUN_NAME}.manifest.json"
# manifest_path.write_text(json.dumps(manifest, indent=2, default=str) + "\n")

## What comes next

Gate: does the water-quality record support formal analysis, or is it contextual? Notebook 08 reports whichever answer this produces.